# Environment

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import re

from pathlib import Path
from typing import List, Tuple, Dict

import matplotlib.pyplot as plt

import os

In [ ]:
wd = os.getcwd()
print(f"El directorio de trabajo actual es: {wd}")

source_tables_path = '/home/alvaro.ritore.ext/projects/trIAje'
os.chdir(source_tables_path)

wd = os.getcwd()
print(f"El directorio de trabajo actual es: {wd}")

In [ ]:
# test for pushing from terminal

## Functions

In [ ]:
# Function to extracts information on nº of id rows from a df
def summarize_demandapk_nuhsa(df: pd.DataFrame):

    # --- 1) Basic counts ---
    n_rows = len(df)
    n_demandapk = df["DEMANDAPK"].nunique()
    n_nuhsa = df["IDTARJETASANITARIA"].nunique()
    n_nuhsa_nan = df["IDTARJETASANITARIA"].isna().sum()

    # --- 2) Unique (DEMANDAPK, NUHSA) pairs (exclude rows with null NUHSA) ---
    unique_pairs = (
        df[["DEMANDAPK", "IDTARJETASANITARIA"]]
        .dropna(subset=["IDTARJETASANITARIA"])
        .drop_duplicates()
    )
    n_unique_pairs = len(unique_pairs)

    # --- Final print in one block ---
    print("\n--- Summary of DEMANDAPK & NUHSA ---")
    print(f"Nº of rows: {n_rows:,}")
    print(f"Nº of unique DEMANDAPK: {n_demandapk:,}")
    print(f"Nº of unique valid NUHSA: {n_nuhsa:,}")
    print(f"Nº of NaN in NUHSA: {n_nuhsa_nan:,}")
    print(f"Unique (DEMANDAPK, NUHSA) pairs (NUHSA not null): {n_unique_pairs:,}")


# Data Loading

## EDATOSCASO

In [ ]:
df_edatoscaso = pq.read_table('data/20_24_CCR_HIST_EDATOS(1)_anonimizado.parquet').to_pandas()

df_edatoscaso.info()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# La anonimización convirtió todos los valores nan en un hash concreto. Se pasan a nan de vuelta
id_tarjeta_deseado = '81SNNM0DHR'  # Asegúrate de que el ID sea una cadena si así está en tu DataFrame
# Utiliza .loc para seleccionar las filas donde la condición se cumple y asigna NaN
df_edatoscaso.loc[df_edatoscaso['IDTARJETASANITARIA'] == id_tarjeta_deseado, 'IDTARJETASANITARIA'] = np.nan
# Para verificar el cambio, puedes contar los NaN en la columna
num_nan = df_edatoscaso['IDTARJETASANITARIA'].isnull().sum()
print(f"\nNúmero de NaN en IDTARJETASANITARIA después de la conversión: {num_nan}")

print("Number of unique patient id is: ", df_edatoscaso['IDTARJETASANITARIA'].nunique())

In [ ]:
print(df_edatoscaso['IDCODIGORESOLUCION'].value_counts())

In [ ]:
df_edatoscaso.head(100)
# df_edatoscaso.head(100).to_csv('export_tables/edatoscaso.csv', sep = ',', index=False)

## Llamadas

In [ ]:
df_llamadas_2020 = pq.read_table('data/2020_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2021 = pq.read_table('data/2021_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2022 = pq.read_table('data/2022_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2023 = pq.read_table('data/2023_CCR_HIST_OtrasTablas.parquet').to_pandas()
df_llamadas_2024 = pq.read_table('data/2024_CCR_HIST_OtrasTablas.parquet').to_pandas()

# Concatenate all tables
lista_dfs = [df_llamadas_2020, df_llamadas_2021, df_llamadas_2022, df_llamadas_2023, df_llamadas_2024]
df_llamadas = pd.concat(lista_dfs, ignore_index=True)
print(f"The size of df_llamadas is: {df_llamadas.shape}")

df_llamadas.info()

# Variables to drop
df_llamadas_var_drop = ['FECHADEMANDA', 'COORDX', 'COORDY', 'DIRPACIENTEIDPROVINCIA', 
                        'DIRPACIENTEIDMUNICIPIO', 'DIRPACIENTEIDLOCALIDAD', 'DIRPACIENTELOCALIDAD']
df_llamadas_reduced = df_llamadas.drop(columns = df_llamadas_var_drop)

df_llamadas_reduced.info()



In [ ]:
df_llamadas_reduced.head()

## ASISTENCIA

In [ ]:
df_asistencia = pd.read_csv('data/dbo.ASISTENCIA.csv', sep = ';')
df_asistencia.info()
df_asistencia.head()


## DEMANDA

In [ ]:
df_demanda = pd.read_csv('data/dbo.DEMANDA.csv', sep = ';')
df_demanda.info()
df_demanda.head()

## Preguntas triaje

In [ ]:
preguntas_triaje = pd.read_excel('data/preguntas_triaje/2001_preg.xlsx')

print(preguntas_triaje.head(10))
print(preguntas_triaje.info())
print(preguntas_triaje.shape[0])

# Data Filtering
## Variables to drop

In [ ]:
# IDTIPODEMANDA: It doesn't provide any information of interest. All values are equal to 'A'
print(df_edatoscaso['IDTIPODEMANDA'].value_counts())

# IDTIPODEMANDA1 and IDTIPODEMANDA2: additional information outside of the scope
# HORAENTRADACONTACTO: information on date and time are contained in other columns

# Columns to drop
columns_to_drop = ['IDTIPODEMANDA', 'IDTIPODEMANDA2', 'IDTIPODEMANDA3', 'HORAENTRADACONTACTO', 
                   'IDMOTIVOEXCLUSION', 'IDUNIDADRECURSO']

df_edatoscaso_reduced = df_edatoscaso.drop(columns = columns_to_drop)
df_edatoscaso_reduced.info()

In [ ]:
df_edatoscaso_reduced.head(100)

In [ ]:
# Before excluding
print(f"The number of rows before applying any exclusion criteria is: {len(df_edatoscaso_reduced):,}")

df_edt_filtered = df_edatoscaso_reduced.copy()

## Tipo Demanda

In [ ]:
id_demanda_triaje = [36, 58, 16, 23, 54]
df_edt_filtered = df_edt_filtered[df_edt_filtered['IDTIPODEMANDA1'].isin(id_demanda_triaje)]

print(df_edt_filtered['IDTIPODEMANDA1'].value_counts())

print("\nAfter filtering for demand types")
summarize_demandapk_nuhsa(df_edt_filtered)

## Extraction of unique ICD by Motivo de demanda

In [ ]:
# from pathlib import Path
# import pandas as pd
# import re

# --- Config ---
demand_col = "IDTIPODEMANDA1"
icd_cols = ["JUICIOCLINICO1", "JUICIOCLINICO2", "JUICIOCLINICO3"]
out_dir = Path("export_tables/icd_unique")
out_dir.mkdir(parents=True, exist_ok=True)

# Map of demand groups -> list of demand codes
demand_groups = {
    "A36_A58_unconsciousness_cardiac_arrest": [36, 58],
    "A16_dyspnea_respiratory_difficulty": [16],
    "A23_non_traumatic_chest_pain": [23],
    "A54_stroke_cerebrovascular_accident": [54],
}

def normalize_icd(x):
    """Return cleaned ICD as string (keep dots), or <NA> if missing/empty."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "na", "none"}:
        return pd.NA
    s = s.replace(",", ".")          # normalize decimal comma
    s = re.sub(r"\s+", "", s)        # remove inner spaces
    return s.upper()                 # ICDs may include letters

def export_unique_icd_by_demand(
    df: pd.DataFrame,
    demand_col: str,
    icd_cols: list[str],
    groups_map: dict[str, list[int]],
    out_dir: Path
):
    """
    For each demand group, filter rows and export a CSV with unique ICDs
    found across JUICIOCLINICO1/2/3. NaNs and empty strings are removed.
    """
    for file_stem, codes in groups_map.items():
        # 1) Filter by demand codes
        subset = df[df[demand_col].isin(codes)].copy()
        if subset.empty:
            print(f"[warn] No rows for {file_stem} (codes={codes}). Skipping.")
            continue

        # 2) Normalize ICD columns and force pandas StringDtype
        icd_clean = subset[icd_cols].map(normalize_icd)
        # 3) Flatten to one column (string dtype), drop NA, deduplicate, sort
        icd_series = (
            icd_clean.melt(value_vars=icd_cols, value_name="ICD")["ICD"]
            .astype("string")
            .dropna()
            .drop_duplicates()
            .sort_values(ignore_index=True)
        )

        # 4) Save to CSV (single column "ICD")
        out_path = out_dir / f"{file_stem}.csv"
        icd_series.to_frame(name="ICD").to_csv(out_path, index=False)
        print(f"Saved {len(icd_series):,} unique ICD codes -> {out_path}")

# --- Run ---
export_unique_icd_by_demand(
    df=df_edt_filtered,
    demand_col=demand_col,
    icd_cols=icd_cols,
    groups_map=demand_groups,
    out_dir=out_dir,
)


## Extraction of nº of IDCODIGORESOLUCION

In [ ]:
# Starting point: a Series of counts (from value_counts)
# e.g. counts = df_duplicates_demandapk['IDCODIGORESOLUCION'].value_counts()
counts = (
    df_edt_filtered["IDCODIGORESOLUCION"]
        .astype("string").str.strip().str.upper()
        .value_counts()
)

# Make a sortable table: split numeric prefix and trailing letters
sorted_tbl = (
    counts.rename_axis("code").reset_index(name="count")
)

# Extract numeric part and alphabetical suffix (both optional)
extracted = sorted_tbl["code"].str.extract(r'^(?P<num>\d+)(?P<suffix>[A-Z]*)$')
sorted_tbl["num"] = pd.to_numeric(extracted["num"], errors="coerce")   # NaN if no leading digits
sorted_tbl["suffix"] = extracted["suffix"].fillna("")

# Sort: numeric asc, then suffix asc; put any non-matching codes at the end
sorted_tbl = sorted_tbl.sort_values(
    by=["num", "suffix", "code"],
    ascending=[True, True, True],
    na_position="last",
    kind="mergesort"  # stable
)

# If you want the same shape as value_counts() but sorted by the custom order:
counts_sorted = sorted_tbl.set_index("code")["count"]
print(counts_sorted)

## NaN in NUHSA in CODIGOSRESOLUCION to Hospital

In [ ]:
# Target resolution codes (strings preserve alphanumerics)
hospital_codes = ["12", "18A", "18B", "23", "24", "26", "26B",
                  "35", "35B", "36", "36B", "36C", "38A", "38B",
                  "43", "44", "45"]

mask = df_edt_filtered["IDCODIGORESOLUCION"].isin(hospital_codes)
df_filtered_by_resolution = df_edt_filtered.loc[mask].copy()

print(f"Rows matching target resolution codes: {len(df_filtered_by_resolution):,}")
print(f"Unique NUHSA: {df_filtered_by_resolution['IDTARJETASANITARIA'].nunique():,}")
print(f"NaN IDTARJETASANITARIA in this subset: {df_filtered_by_resolution['IDTARJETASANITARIA'].isna().sum():,}")

# Breakdown by resolution code (sorted by code)
print(df_filtered_by_resolution['IDCODIGORESOLUCION'].value_counts())


## Duplicates

In [ ]:
print(f"\nNº of rows: {len(df_edt_filtered):,}")
print(f"Nº of unique DEMANDAPK: {df_edt_filtered['DEMANDAPK'].nunique():,}")
print(f"Nº of unique valid NUHSA: {df_edt_filtered['IDTARJETASANITARIA'].nunique():,}")
print(f"Nº of NaN in NUHSA: {df_edt_filtered['IDTARJETASANITARIA'].isna().sum():,}")

# Count rows per DEMANDAPK
per_demand_counts = (
    df_edt_filtered.groupby("DEMANDAPK", dropna=False)
                   .size()
                   .rename("n_rows")
                   .reset_index()
)

# Distribution: how many DEMANDAPK have 1 row, 2 rows, ...
n_rows_distribution = (
    per_demand_counts["n_rows"]
    .value_counts()
    .sort_index()
    .rename_axis("n_rows_per_DEMANDAPK")
    .reset_index(name="n_DEMANDAPK")
)

print("\nDistribution: number of rows per DEMANDAPK -> how many DEMANDAPK have that many rows")
print(n_rows_distribution)

# # (Optional) List DEMANDAPK with duplicates (>1 row)
# duplicates_list = per_demand_counts.query("n_rows > 1").sort_values("n_rows", ascending=False)
# print("DEMANDAPK with duplicates (more than 1 row):")
# print(duplicates_list.head(20))  # change/remove head() to see more


In [ ]:
# Count how many distinct NUHSA appear more than once (excluding NaN)
nuhsa_col = "IDTARJETASANITARIA"

# Use string to preserve leading zeros and strip accidental spaces
nuhsa = df_edt_filtered[nuhsa_col].astype("string").str.strip()

# Frequency of non-null NUHSA
nuhsa_counts = nuhsa.dropna().value_counts()

# NUHSA that occur >1 time
nuhsa_repeated = nuhsa_counts[nuhsa_counts > 1]

# Metrics
n_unique_repeated = nuhsa_repeated.size                 # distinct NUHSA with >1 appearances
n_rows_repeated = nuhsa_repeated.sum()                  # total rows belonging to those NUHSA
n_extra_duplicate_rows = (nuhsa_repeated - 1).sum()     # extra rows beyond the first occurrence

print(f"Distinct NUHSA appearing >1 time: {n_unique_repeated:,}")
print(f"Total rows with those NUHSA: {n_rows_repeated:,}")
print(f"Extra duplicate rows beyond firsts: {n_extra_duplicate_rows:,}")

# Table of repeated NUHSA with counts (sorted desc)
repeated_table = (
    nuhsa_repeated.reset_index()
    .rename(columns={"index": "NUHSA", 0: "count"})
)
# print(repeated_table.head(20))  # show top 20


In [ ]:
# 1) Keep only DEMANDAPK that appear exactly twice
df_two_rows = df_edt_filtered[
    df_edt_filtered.groupby("DEMANDAPK")["DEMANDAPK"].transform("size") == 2
].copy()

# --- NEW BLOCK: quality check on NUHSA (IDTARJETASANITARIA) within DEMANDAPK with 2 rows ---
# For each DEMANDAPK (with exactly 2 rows), count how many NaN NUHSA it has.
nuhsa_nulls_per_demand = (
    df_two_rows.groupby("DEMANDAPK")["IDTARJETASANITARIA"]
               .apply(lambda s: s.isna().sum())
)

n_demand_with_any_nan = (nuhsa_nulls_per_demand > 0).sum()
n_demand_exactly_one_nan = (nuhsa_nulls_per_demand == 1).sum()
n_demand_both_nan = (nuhsa_nulls_per_demand == 2).sum()
n_total_demands_twice = df_two_rows["DEMANDAPK"].nunique()

print(f"DEMANDAPK with exactly 2 rows: {n_total_demands_twice:,}")
print(f"DEMANDAPK with any NaN in IDTARJETASANITARIA: {n_demand_with_any_nan:,}")
print(f" ├─ exactly one NaN: {n_demand_exactly_one_nan:,}")
print(f" └─ both rows NaN: {n_demand_both_nan:,}")

# 2) Filter by the specified hospital codes (no normalization assumed)
hospital_codes = ["12", "18A", "18B", "23", "24", "26", "26B",
                  "35", "35B", "36", "36B", "36C", "38A", "38B",
                  "43", "44", "45"]

mask_codes = df_two_rows["IDCODIGORESOLUCION"].isin(hospital_codes)
df_result = df_two_rows.loc[mask_codes].copy()

# 3) Count rows with NaN in IDTARJETASANITARIA within the filtered result
total_rows = len(df_result)
nuhsa_nan = df_result["IDTARJETASANITARIA"].isna().sum()

print(f"Rows with DEMANDAPK==2 and target resolution codes: {total_rows:,}")
print(f"Rows with NaN IDTARJETASANITARIA in that subset: {nuhsa_nan:,}")

# (Optional) breakdown by resolution code in the final subset
# print(df_result['IDCODIGORESOLUCION'].value_counts().sort_index())


In [ ]:
# 1) Keep only DEMANDAPK that appear exactly twice
df_two_rows = df_edt_filtered[
    df_edt_filtered.groupby("DEMANDAPK")["DEMANDAPK"].transform("size") == 2
].copy()

# 2) DEMANDAPK with exactly one unique non-null NUHSA
#    (includes cases with one NUHSA + one NaN; excludes both-NaN)
unique_non_null_nuhsa = (
    df_two_rows.groupby("DEMANDAPK")["IDTARJETASANITARIA"]
               .apply(lambda s: s.astype("string").dropna().nunique())
)
demands_with_one_nuhsa = unique_non_null_nuhsa[unique_non_null_nuhsa == 1].index

df_one_nuhsa = df_two_rows[df_two_rows["DEMANDAPK"].isin(demands_with_one_nuhsa)].copy()

# 3) Filter by the specified hospital codes
hospital_codes = ["12", "18A", "18B", "23", "24", "26", "26B",
                  "35", "35B", "36", "36B", "36C", "38A", "38B",
                  "43", "44", "45"]

mask_codes = df_one_nuhsa["IDCODIGORESOLUCION"].isin(hospital_codes)
df_subset_one_nuhsa_codes = df_one_nuhsa.loc[mask_codes].copy()

# 4) Count rows with NaN NUHSA in the final subset
total_rows = len(df_subset_one_nuhsa_codes)
nuhsa_nan_rows = df_subset_one_nuhsa_codes["IDTARJETASANITARIA"].isna().sum()

print(f"DEMANDAPK with exactly 2 rows: {df_two_rows['DEMANDAPK'].nunique():,}")
print(f"DEMANDAPK with exactly one unique non-null NUHSA: {len(demands_with_one_nuhsa):,}")
print(f"Rows after code filter: {total_rows:,}")
print(f"Rows with NaN IDTARJETASANITARIA in that subset: {nuhsa_nan_rows:,}")

# (Optional) save the subset
# from pathlib import Path
# out_path = Path("export_tables/subset_one_nuhsa_codes.csv")
# out_path.parent.mkdir(parents=True, exist_ok=True)
# df_subset_one_nuhsa_codes.to_csv(out_path, index=False)
# print(f"Saved -> {out_path}")


In [ ]:
# Select rows that are duplicates in the 'DEMANDAPK' column
# keep=False marks all occurrences of a duplicated value as True
# so you get all rows that are part of a duplicate set.
df_edt_duplicates_demandapk = df_edt_filtered[df_edt_filtered['DEMANDAPK'].duplicated(keep=False)].sort_values(by='DEMANDAPK')


# Optionally, you can also check the shape to see how many duplicated rows there are
print(f"\nNumber of duplicated rows based on 'DEMANDAPK': {df_edt_duplicates_demandapk.shape[0]:,}")
print(f"\nNumber of unique 'DEMANDAPK': {df_edt_duplicates_demandapk['DEMANDAPK'].nunique():,}\n")


print(df_edt_duplicates_demandapk['IDCODIGORESOLUCION'].value_counts())

## Duplicates by DEMANDAPK with 2 rows. Unique IDTIPORECURSO-IDCODIGORESOLUCION pairs

In [ ]:
# --- 1) DEMANDAPK con exactamente 2 registros ---
freq = df_edt_filtered["DEMANDAPK"].value_counts()
pair_ids = freq[freq == 2].index

df_pairs = df_edt_filtered[df_edt_filtered["DEMANDAPK"].isin(pair_ids)].copy()

summarize_demandapk_nuhsa(df_pairs)

# --- 2) Crear la dupla fila a fila ---
df_pairs["DUPLA"] = (
    df_pairs["IDTIPORECURSO"].astype("string").str.strip().fillna("NA") + " - " +
    df_pairs["IDCODIGORESOLUCION"].astype("string").str.strip().fillna("NA")
)

# --- 3) Agrupar por DEMANDAPK y formar combinación de las 2 duplas ---
def make_pair(x):
    duplas = sorted(x.tolist())   # ordenamos para que (A,B) = (B,A)
    return tuple(duplas)

pairs_combo = (
    df_pairs.groupby("DEMANDAPK")["DUPLA"]
            .apply(make_pair)
            .value_counts()
            .reset_index()
)

pairs_combo.columns = ["COMBINACION_PAREJA", "n"]

# --- 4) Mostrar en orden descendente ---
print(f"Número total de DEMANDAPK con exactamente 2 duplicados: {len(pair_ids):,}\n")
print("Distribución de combinaciones de duplas en parejas:")
print(pairs_combo.head(50))

## Grouping of Duplicates

In [ ]:
# Deduplicate DEMANDAPK with exactly two rows, under column-aware rules

# -----------------------------
# Configuration
# -----------------------------
ID_COL = "DEMANDAPK"
NUHSA_COL = "IDTARJETASANITARIA"

# Columns whose two values should be combined into a single string
# when both rows remain but NUHSA is the same. Adjust as needed.
COMBINE_COLS: List[str] = [
    "IDTIPORECURSO",
    "IDCODIGORESOLUCION",
    "JUICIOCLINICO1",
    "JUICIOCLINICO2",
    "JUICIOCLINICO3",
    "PRIORIDAD", 
    "PRIORIDADTIPORECURSO", 
    "IDTIPORECURSOAGR"
]

# Separator used when combining two distinct values in COMBINE_COLS
COMBINE_SEP = " | "

# -----------------------------
# Helpers
# -----------------------------

def _is_valid_nuhsa(v) -> bool:
    """A NUHSA is valid if it's not NA and not empty after strip.
    Does not normalize case; preserves original forms.
    """
    if pd.isna(v):
        return False
    s = str(v)
    return s.strip() != ""

def _is_missing(v) -> bool:
    """Generic missingness check for combining:
    - True for pandas NA/NaN/None
    - True for empty/whitespace-only strings
    - False otherwise
    """
    if pd.isna(v):
        return True
    try:
        return str(v).strip() == ""
    except Exception:
        return False


def _combine_two_values(v1, v2, sep: str = COMBINE_SEP) -> object:
    """Combine two cell values for COMBINE_COLS with rules:
    - If one side is missing (NaN/None or blank/whitespace), return the other side.
    - If both present and equal after strip (case-sensitive), return v1 (preserve original form).
    - Else, return f"{v1}{sep}{v2}".
    """
    m1, m2 = _is_missing(v1), _is_missing(v2)
    if m1 and m2:
        return pd.NA
    if m1:
        return v2
    if m2:
        return v1
    s1, s2 = str(v1).strip(), str(v2).strip()
    if s1 == s2:
        return v1
    return f"{v1}{sep}{v2}"


# -----------------------------
# Row-combining helper
# -----------------------------

def _combine_selected_columns(base_row: pd.Series, other_row: pd.Series, 
                              combine_cols: List[str], sep: str = COMBINE_SEP) -> pd.Series:
    """
    Return a copy of base_row where each column in combine_cols is set to the
    combination of base_row[col] and other_row[col] using _combine_two_values.
    """
    out = base_row.copy()
    for c in combine_cols:
        if c in out.index and c in other_row.index:
            out[c] = _combine_two_values(base_row[c], other_row[c], sep=sep)
    return out

# -----------------------------
# Core logic
# -----------------------------

def resolve_two_row_duplicates(
    df: pd.DataFrame,
    id_col: str = ID_COL,
    nuhsa_col: str = NUHSA_COL,
    combine_cols: List[str] = COMBINE_COLS,
    combine_sep: str = COMBINE_SEP,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Resolve duplicates for DEMANDAPK that appear exactly twice, under rules:
      A) If exactly one row has a valid NUHSA -> keep that row **and combine** COMBINE_COLS with the other row (for traceability).
      B) If both NUHSA are valid and equal -> collapse to a single row, **combining COMBINE_COLS** and filling missing non-combine fields from the other row.
      C) If both NUHSA are valid and different -> keep both rows unchanged.
      D) If both NUHSA are NA/empty -> keep both rows unchanged (explicit).

    Returns
    -------
    resolved_pairs : DataFrame
        The deduplicated/combined result **only** for DEMANDAPK with exactly two rows.
    summary : DataFrame
        Per-group case classification and counts (#rows in output per DEMANDAPK).
    """
    # 1) Identify DEMANDAPK with exactly two rows
    freq = df[id_col].value_counts(dropna=False)
    pair_ids = freq[freq == 2].index
    df_pairs = df[df[id_col].isin(pair_ids)].copy()

    # Safety: ensure we truly have groups of size 2
    assert df_pairs.groupby(id_col).size().min() == 2 and df_pairs.groupby(id_col).size().max() == 2, \
        "This function expects exactly two rows per group in df_pairs."

    records = []  # collected output rows
    summary_rows = []

    for gid, g in df_pairs.groupby(id_col, sort=False):
        g = g.sort_index()  # stable order by original index
        r1, r2 = g.iloc[0].copy(), g.iloc[1].copy()

        v1, v2 = r1[nuhsa_col], r2[nuhsa_col]
        valid1, valid2 = _is_valid_nuhsa(v1), _is_valid_nuhsa(v2)

        if valid1 and not valid2:
            case = "one_valid_keep_row1"
            # Keep row1 but also combine selected columns with row2
            out = _combine_selected_columns(r1, r2, combine_cols, combine_sep)
            out_rows = 1
            records.append(out)
        elif valid2 and not valid1:
            case = "one_valid_keep_row2"
            # Keep row2 but also combine selected columns with row1
            out = _combine_selected_columns(r2, r1, combine_cols, combine_sep)
            out_rows = 1
            records.append(out)
        elif valid1 and valid2 and str(v1).strip() == str(v2).strip():
            case = "both_valid_equal_collapse"
            # Start from first row and combine selected columns
            out = _combine_selected_columns(r1, r2, combine_cols, combine_sep)
            # For non-combine columns (excluding id and nuhsa), fill NA from the other row
            skip = set([id_col, nuhsa_col] + combine_cols)
            for c in g.columns:
                if c in skip:
                    continue
                if pd.isna(out[c]) and not pd.isna(r2[c]):
                    out[c] = r2[c]
            out_rows = 1
            records.append(out)
        else:
            # (valid1 and valid2 and different) OR (both invalid)
            case = "keep_both"
            out_rows = 2
            records.extend([r1, r2])

        summary_rows.append({
            id_col: gid,
            "case": case,
            "output_rows": out_rows,
            "nuhsa_r1": v1,
            "nuhsa_r2": v2,
        })

    df_grouped_2rows = pd.DataFrame(records).reset_index(drop=True)
    df_grouped_2rows_summary = pd.DataFrame(summary_rows)

    return df_grouped_2rows, df_grouped_2rows_summary


# -----------------------------
# Example usage
# -----------------------------
df_grouped_2rows, df_grouped_2rows_summary = resolve_two_row_duplicates(df_edt_filtered[:10000])
print(df_grouped_2rows_summary["case"].value_counts())
print(df_grouped_2rows.shape, df_grouped_2rows_summary.shape)

# # If you want to stitch back with the rest of the dataset (non-two-row groups):
# freq_all = df[ID_COL].value_counts(dropna=False)
# other_ids = freq_all.index.difference(freq_all[freq_all == 2].index)
# df_rest = df[df[ID_COL].isin(other_ids)].copy()
# df_final = pd.concat([df_rest, resolved], ignore_index=True)

# # Optional: keep column order identical to original
# df_final = df_final[df.columns]


In [ ]:
summarize_demandapk_nuhsa(df_grouped_2rows)

In [ ]:
df_grouped_2rows.head(100)

### More than one NUHSA for the same DEMANDAPK

In [ ]:
# 1. Agrupar por 'DEMANDAPK' y contar los valores únicos de 'IDTARJETASANITARIA' en cada grupo.
#    .nunique() EXCLUYE NaN por defecto.
unique_id_per_demandapk = df_edt_duplicates_demandapk.groupby('DEMANDAPK')['IDTARJETASANITARIA'].nunique()
unique_id_per_demandapk.info()

# 2. Filtrar esta Serie para quedarnos solo con los DEMANDAPK que tienen más de 1 IDTARJETASANITARIA único.
#    Esto significa que la misma DEMANDAPK está asociada con dos o más IDTARJETASANITARIA VÁLIDOS y distintos.
demandapk_with_multiple_ids = unique_id_per_demandapk[unique_id_per_demandapk > 1]

# 3. Contar cuántos DEMANDAPK cumplen esta condición.
print("Número de DEMANDAPK únicos con más de un IDTARJETASANITARIA distinto (excluyendo NaN): ", 
      len(demandapk_with_multiple_ids))


In [ ]:
# Opcional: Para ver un ejemplo de los datos completos de esas DEMANDAPK específicas para inspección
# (útil para entender por qué se marcan como múltiples IDs)
filtered_data_multiple_ids = df_edt_duplicates_demandapk[df_edt_duplicates_demandapk['DEMANDAPK']
                                                         .isin(demandapk_with_multiple_ids.index)]
print("IDTIPODEMANDA1 in DEMANDAPK with >1 NUHSA:\n", filtered_data_multiple_ids['IDTIPODEMANDA1'].value_counts())

print("\nPrimeras filas de los datos para DEMANDAPK con múltiples IDTARJETASANITARIA para inspección:")
filtered_data_multiple_ids.sort_values(by='DEMANDAPK').head(10)

In [ ]:
demandapk_with_multiple_ids.head(100)

# Data cleaning

In [ ]:
df_edt_cleaned = df_edt_filtered.copy()

## AGE

In [ ]:
df_edt_cleaned['EDAD'].hist(bins=20, range=(0, 100), color='skyblue', edgecolor='black')
plt.title('Histograma de Edad')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.grid(True, linestyle='--', alpha=0.6) # Añade una cuadrícula tenue
plt.show()

df_edt_cleaned.boxplot(column=['EDAD'], by='IDTIPODEMANDA1')
plt.title('Boxplot de Edad')
plt.ylabel('Edad')
plt.show()

num_vacios = (df_edt_cleaned['FNACIMIENTO'] == '').sum()
print(f"El número de valores vacíos ('') en la columna 'FNACIMIENTO' es: {num_vacios}")

## SEX

In [ ]:
print("Value counts of sex column before data cleaning:")
print(df_edt_cleaned['SEXO'].value_counts())

# Removing non-valid sex values and encoding values 
df_edt_cleaned = df_edt_cleaned[df_edt_cleaned['SEXO'].isin(['M', 'H'])].copy()
map_sex = {'M': 1, 'H': 0}
df_edt_cleaned.loc[:, 'SEXO'] = df_edt_cleaned['SEXO'].map(map_sex)

print("\nValue counts of sex column after data cleaning:")
print(df_edt_cleaned['SEXO'].value_counts())

print(f"\nAfter excluding by non-valid sex values, the number of rows is: {df_edt_cleaned.shape[0]:,}")

## Fecha demanda

In [ ]:
print(f"\nThe number of nan in FECHADEMANDA is {df_edt_cleaned['FECHADEMANDA'].isnull().sum()}")

df_edt_cleaned['FECHADEMANDA'] = pd.to_datetime(df_edt_cleaned['FECHADEMANDA'], format='%Y%m%d')




# Creation of new variables

In [ ]:
df_edt_new_var = df_edt_cleaned.copy()

print(f"\nThe number of nan in PRIORIDAD is {df_edt_new_var['PRIORIDAD'].isnull().sum()}")

# P1_predicted is the priority predicted during the call, assigned by the system
# 1 is maximum priority, 0 is non-maximum priority
df_edt_new_var['P1_predicted'] = np.where(df_edt_new_var['PRIORIDAD']==1, 1, 0)

# Create the day of the week (number, Monday=0, Sunday=6) variable
df_edt_new_var['day_week'] = df_edt_new_var['FECHADEMANDA'].dt.dayofweek

# Create the month (number) variable
df_edt_new_var['month'] = df_edt_new_var['FECHADEMANDA'].dt.month

# Create the year variable
df_edt_new_var['year'] = df_edt_new_var['FECHADEMANDA'].dt.year

# Display the new columns and the original date column
print(df_edt_new_var[['FECHADEMANDA', 'day_week', 'month', 'year']].head())


## DATA QUALITY

In [ ]:
print(f"\nThe number of nan in IDTIPODEMANDA1 is {df_edt_cleaned['IDTIPODEMANDA1'].isnull().sum()}")

df_edt_new_var.head(50)
df_edt_new_var.head(100).to_csv('export_tables/triaje_table.csv', sep = ',', index=False)

In [ ]:
id_tarjeta_deseado = '81SNNM0DHR'  # Asegúrate de que el ID sea una cadena si así está en tu DataFrame

tabla_filtrada = df_edt_new_var[df_edt_new_var['IDTARJETASANITARIA'] == id_tarjeta_deseado]
tabla_filtrada.head(10)

In [ ]:
print(f"El NUHSA {id_tarjeta_deseado} aparece en el dataset {len(tabla_filtrada)} veces")

In [ ]:
print("IDTIPORECURSO", df_edt_cleaned['IDTIPORECURSO'].value_counts())